# 📘 HTTP/REST APIs for Data Engineering
## Databricks Notebook — API Ingestion Mastery

**What you'll learn:**
- HTTP fundamentals: methods, status codes, headers
- The `requests` library: GET, POST, sessions, auth
- Pagination patterns: offset, cursor, link-header
- Error handling & retries with exponential backoff
- Batch ingestion: API → Bronze → Silver
- Webhook processing, caching, OAuth2

**Prerequisites:** Notebooks 01-11

**Key rules:**
- Always set `timeout` on every request
- Never hardcode API keys — use secrets
- Always check `status_code` before `.json()`

---

---
# 📗 Section 1: Why APIs Matter for Data Engineering

**Common API patterns in enterprise DE:**
1. **Ingestion:** Pull data from SaaS platforms (Salesforce, Stripe, HubSpot)
2. **Orchestration:** Trigger Databricks jobs via REST API
3. **Monitoring:** Send alerts to Slack/PagerDuty
4. **Enrichment:** Geocoding, currency conversion, weather data
5. **Webhooks:** Receive real-time events from external systems

**Pipeline pattern:**
```
REST API → [Fetch + Paginate] → Bronze (raw JSON) → Silver (parsed, validated)
```

---
# 📗 Section 2: HTTP Fundamentals

| Method | Purpose | Idempotent? | Body? |
|---|---|---|---|
| GET | Read data | ✅ Yes | ❌ No |
| POST | Create data | ❌ No | ✅ Yes |
| PUT | Replace data | ✅ Yes | ✅ Yes |
| PATCH | Partial update | ❌ No | ✅ Yes |
| DELETE | Remove data | ✅ Yes | ❌ No |

**Status codes:**
- `2xx` — Success (200 OK, 201 Created, 204 No Content)
- `3xx` — Redirect (301 Moved, 304 Not Modified)
- `4xx` — Client error (400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 429 Rate Limited)
- `5xx` — Server error (500 Internal, 502 Bad Gateway, 503 Unavailable)

In [ ]:
import requests
import time

# Basic GET request to a public API
url = "https://jsonplaceholder.typicode.com/posts/1"
response = requests.get(url, timeout=30)

print(f"Status: {response.status_code}")
print(f"Headers: {dict(list(response.headers.items())[:3])}")
print(f"Content-Type: {response.headers.get('Content-Type')}")

if response.status_code == 200:
    data = response.json()
    print(f"\nResponse data:")
    print(f"  id: {data['id']}")
    print(f"  title: {data['title'][:50]}...")

---
# 📗 Section 3: The requests Library

In [ ]:
# GET with query parameters
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 1, "_limit": 5},
    timeout=30
)
print(f"URL sent: {response.url}")
print(f"Status: {response.status_code}")
posts = response.json()
print(f"Got {len(posts)} posts")
for post in posts[:3]:
    print(f"  [{post['id']}] {post['title'][:40]}...")

In [ ]:
# POST request (create data)
new_post = {
    "title": "Data Engineering Best Practices",
    "body": "Always validate your data at pipeline boundaries.",
    "userId": 1
}

response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=new_post,  # automatically sets Content-Type: application/json
    timeout=30
)
print(f"Status: {response.status_code}")  # 201 Created
print(f"Created: {response.json()}")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Fetch & Parse API Data
# ============================================
# 1. GET all users from https://jsonplaceholder.typicode.com/users
# 2. Check status code
# 3. Parse JSON response
# 4. Print each user's name, email, and company name
# 5. Count users per city
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=30)
assert response.status_code == 200, f"Failed: {response.status_code}"

users = response.json()
print(f"Got {len(users)} users:\n")
city_counts = {}
for user in users:
    print(f"  {user['name']} | {user['email']} | {user['company']['name']}")
    city = user['address']['city']
    city_counts[city] = city_counts.get(city, 0) + 1

print(f"\nUsers per city: {city_counts}")

---
# 📗 Section 4: Authentication Patterns

In [ ]:
# Simulated OAuth2 client credentials flow
import time

class OAuth2Client:
    """OAuth2 client credentials flow for service-to-service auth."""
    
    def __init__(self, token_url, client_id, client_secret):
        self.token_url = token_url
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = None
        self.token_expiry = 0
    
    def get_token(self):
        """Get or refresh access token."""
        if self.access_token and time.time() < self.token_expiry:
            return self.access_token
        
        # In production: POST to token_url with credentials
        # response = requests.post(self.token_url, data={
        #     "grant_type": "client_credentials",
        #     "client_id": self.client_id,
        #     "client_secret": self.client_secret
        # })
        # token_data = response.json()
        
        # Simulated response
        self.access_token = "eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.simulated"
        self.token_expiry = time.time() + 3600  # 1 hour
        print(f"  🔑 Token refreshed (expires in 3600s)")
        return self.access_token
    
    def authenticated_request(self, method, url, **kwargs):
        """Make an authenticated request with auto-refresh."""
        token = self.get_token()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {token}"
        return requests.request(method, url, headers=headers, timeout=30, **kwargs)

# Demo
client = OAuth2Client("https://auth.example.com/oauth/token", "my_client_id", "my_secret")
token = client.get_token()
print(f"Token: {token[:30]}...")

---
# 📗 Section 5: Session Management

In [ ]:
# requests.Session for connection reuse (3-5x faster for multiple calls)
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def create_robust_session(max_retries=3, backoff_factor=0.5):
    """Create a session with automatic retries."""
    session = requests.Session()
    
    # Configure retry strategy
    retry_strategy = Retry(
        total=max_retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )
    
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    
    # Default headers
    session.headers.update({
        "Accept": "application/json",
        "User-Agent": "DataPipeline/1.0"
    })
    
    return session

# Use session for multiple calls (reuses TCP connection)
session = create_robust_session()
urls = [
    "https://jsonplaceholder.typicode.com/posts/1",
    "https://jsonplaceholder.typicode.com/posts/2",
    "https://jsonplaceholder.typicode.com/posts/3",
]

start = time.time()
for url in urls:
    resp = session.get(url, timeout=30)
    print(f"  {url.split('/')[-1]}: {resp.status_code}")
print(f"Session (3 calls): {time.time()-start:.2f}s")
session.close()

---
# 📗 Section 6: Pagination Patterns

In [ ]:
# Offset-based pagination
def fetch_all_offset(base_url, page_size=10, max_pages=5):
    """Fetch all pages using offset pagination."""
    all_records = []
    page = 1
    
    while page <= max_pages:
        response = requests.get(
            base_url,
            params={"_page": page, "_limit": page_size},
            timeout=30
        )
        
        if response.status_code != 200:
            print(f"  Error on page {page}: {response.status_code}")
            break
        
        data = response.json()
        if not data:  # Empty page = done
            break
        
        all_records.extend(data)
        print(f"  Page {page}: got {len(data)} records (total: {len(all_records)})")
        page += 1
    
    return all_records

# Fetch paginated posts
print("Offset pagination:")
posts = fetch_all_offset("https://jsonplaceholder.typicode.com/posts", page_size=20, max_pages=3)
print(f"Total fetched: {len(posts)} posts")

In [ ]:
# Cursor-based pagination (simulated)
def fetch_all_cursor(base_url, page_size=10):
    """Fetch all pages using cursor-based pagination."""
    all_records = []
    cursor = None
    page = 0
    
    while True:
        params = {"limit": page_size}
        if cursor:
            params["cursor"] = cursor
        
        # Simulated: use offset as cursor proxy
        response = requests.get(
            base_url,
            params={"_start": page * page_size, "_limit": page_size},
            timeout=30
        )
        
        data = response.json()
        if not data:
            break
        
        all_records.extend(data)
        
        # In real APIs: cursor = response.json().get("next_cursor")
        cursor = f"offset_{(page+1) * page_size}"
        page += 1
        
        if page >= 3:  # Safety limit for demo
            break
    
    return all_records

print("\nCursor pagination (simulated):")
results = fetch_all_cursor("https://jsonplaceholder.typicode.com/posts", page_size=20)
print(f"Total fetched: {len(results)} records")

---
# 📗 Section 7: Error Handling & Retries

In [ ]:
# Robust API client with exponential backoff
import time
import random

def api_call_with_retry(url, max_retries=3, base_delay=1.0, timeout=30):
    """Make an API call with exponential backoff retry.
    
    Retries on: 429 (rate limit), 500, 502, 503, 504
    Does NOT retry on: 400, 401, 403, 404 (client errors)
    """
    retryable_codes = {429, 500, 502, 503, 504}
    
    for attempt in range(1, max_retries + 1):
        try:
            start = time.time()
            response = requests.get(url, timeout=timeout)
            duration = time.time() - start
            
            if response.status_code == 200:
                print(f"  ✅ Attempt {attempt}: {response.status_code} ({duration:.2f}s)")
                return response.json()
            elif response.status_code in retryable_codes:
                delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 1)
                print(f"  ⚠️ Attempt {attempt}: {response.status_code} — retrying in {delay:.1f}s")
                if attempt < max_retries:
                    time.sleep(delay)
            else:
                print(f"  ❌ Attempt {attempt}: {response.status_code} — not retryable")
                return None
                
        except requests.exceptions.Timeout:
            print(f"  ⏱️ Attempt {attempt}: Timeout — retrying...")
        except requests.exceptions.ConnectionError:
            print(f"  🔌 Attempt {attempt}: Connection error — retrying...")
    
    print(f"  ❌ All {max_retries} attempts failed")
    return None

# Test with a real API
print("API call with retry:")
result = api_call_with_retry("https://jsonplaceholder.typicode.com/posts/1")
if result:
    print(f"  Got: {result['title'][:40]}...")

---
# 📗 Section 8: Batch Ingestion from APIs

In [ ]:
# Complete API → Bronze → Silver pipeline
import json as json_lib
from datetime import datetime
import pandas as pd

def ingest_api_to_bronze(base_url, pages=3, page_size=20):
    """Fetch paginated API data and store as Bronze records."""
    bronze_records = []
    
    for page in range(1, pages + 1):
        start = time.time()
        response = requests.get(
            base_url,
            params={"_page": page, "_limit": page_size},
            timeout=30
        )
        duration = time.time() - start
        
        bronze_records.append({
            "raw_response": json_lib.dumps(response.json()),
            "ingested_at": datetime.now().isoformat(),
            "source_url": response.url,
            "page_number": page,
            "http_status": response.status_code,
            "response_time_ms": int(duration * 1000),
            "record_count": len(response.json())
        })
        
        print(f"  Page {page}: {response.status_code}, {len(response.json())} records, {duration*1000:.0f}ms")
    
    return bronze_records

# Ingest
print("Ingesting from API:")
bronze_data = ingest_api_to_bronze("https://jsonplaceholder.typicode.com/posts")

# Write to Bronze
bronze_df = spark.createDataFrame(pd.DataFrame(bronze_data))
bronze_df.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.bronze_api_posts")
print(f"\n✅ Bronze: {bronze_df.count()} records written")

In [ ]:
# Transform Bronze → Silver (parse, flatten, validate)
import json as json_lib
import pandas as pd

# Read bronze
bronze_pdf = spark.table("techmart_dw.bronze_api_posts").toPandas()

# Parse and flatten
silver_records = []
for _, row in bronze_pdf.iterrows():
    posts = json_lib.loads(row["raw_response"])
    for post in posts:
        silver_records.append({
            "post_id": post["id"],
            "user_id": post["userId"],
            "title": post["title"].strip(),
            "body": post["body"].strip()[:200],
            "title_length": len(post["title"]),
            "_source_page": row["page_number"],
            "_ingested_at": row["ingested_at"]
        })

# Deduplicate by natural key
silver_pdf = pd.DataFrame(silver_records).drop_duplicates(subset=["post_id"], keep="last")

# Write to Silver
silver_df = spark.createDataFrame(silver_pdf)
silver_df.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.silver_api_posts")
print(f"✅ Silver: {silver_df.count()} unique posts")
silver_df.show(5, truncate=40)

---
# 📗 Section 9: Webhook Simulation

In [ ]:
# Simulate processing webhook events
import json as json_lib
import hashlib
from datetime import datetime

def process_webhook_batch(events):
    """Process a batch of webhook events with idempotency."""
    processed_keys = set()
    results = {"processed": 0, "duplicates": 0, "errors": 0}
    
    for event in events:
        # Idempotency: use event_id to detect duplicates
        idempotency_key = event.get("event_id", hashlib.md5(json_lib.dumps(event, sort_keys=True).encode()).hexdigest())
        
        if idempotency_key in processed_keys:
            results["duplicates"] += 1
            continue
        
        try:
            # Validate required fields
            assert "event_type" in event, "Missing event_type"
            assert "timestamp" in event, "Missing timestamp"
            assert "data" in event, "Missing data"
            
            processed_keys.add(idempotency_key)
            results["processed"] += 1
        except (AssertionError, Exception) as e:
            results["errors"] += 1
    
    return results

# Simulate webhook events (including duplicates)
webhook_events = [
    {"event_id": "evt_001", "event_type": "order.created", "timestamp": "2024-06-15T10:00:00Z", "data": {"order_id": 1001}},
    {"event_id": "evt_002", "event_type": "payment.completed", "timestamp": "2024-06-15T10:01:00Z", "data": {"payment_id": 2001}},
    {"event_id": "evt_001", "event_type": "order.created", "timestamp": "2024-06-15T10:00:00Z", "data": {"order_id": 1001}},  # DUPLICATE
    {"event_id": "evt_003", "event_type": "shipment.shipped", "timestamp": "2024-06-15T10:05:00Z", "data": {"shipment_id": 3001}},
    {"event_type": "invalid.event", "timestamp": "2024-06-15T10:06:00Z"},  # Missing data field
]

results = process_webhook_batch(webhook_events)
print(f"Webhook processing results:")
print(f"  Processed: {results['processed']}")
print(f"  Duplicates skipped: {results['duplicates']}")
print(f"  Errors: {results['errors']}")

---
# 🚀 Section 10: Mini Project — Complete API Ingestion Pipeline

In [ ]:
# Complete pipeline: paginated API → Bronze → Silver with metadata tracking
import json as json_lib
import time
from datetime import datetime
import pandas as pd

class APIIngestionPipeline:
    """Complete API ingestion pipeline with tracking."""
    
    def __init__(self, name, base_url, target_db="techmart_dw"):
        self.name = name
        self.base_url = base_url
        self.target_db = target_db
        self.metadata = {
            "pipeline": name,
            "started_at": datetime.now().isoformat(),
            "pages_fetched": 0,
            "records_ingested": 0,
            "errors": 0
        }
    
    def fetch_page(self, page, page_size=20):
        """Fetch a single page with error handling."""
        try:
            resp = requests.get(
                self.base_url,
                params={"_page": page, "_limit": page_size},
                timeout=30
            )
            resp.raise_for_status()
            return resp.json(), resp.status_code
        except Exception as e:
            self.metadata["errors"] += 1
            return [], 500
    
    def ingest(self, max_pages=5, page_size=20):
        """Run full ingestion."""
        print(f"🚀 Starting pipeline: {self.name}")
        all_bronze = []
        
        for page in range(1, max_pages + 1):
            data, status = self.fetch_page(page, page_size)
            if not data:
                break
            
            all_bronze.append({
                "raw_json": json_lib.dumps(data),
                "page": page,
                "record_count": len(data),
                "status": status,
                "fetched_at": datetime.now().isoformat()
            })
            self.metadata["pages_fetched"] += 1
            self.metadata["records_ingested"] += len(data)
        
        # Write Bronze
        bronze_df = spark.createDataFrame(pd.DataFrame(all_bronze))
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(f"{self.target_db}.bronze_{self.name}")
        
        # Transform to Silver
        silver_records = []
        for row in all_bronze:
            for record in json_lib.loads(row["raw_json"]):
                silver_records.append({
                    "id": record.get("id"),
                    "user_id": record.get("userId"),
                    "title": record.get("title", "").strip(),
                    "_page": row["page"],
                    "_fetched_at": row["fetched_at"]
                })
        
        silver_pdf = pd.DataFrame(silver_records).drop_duplicates(subset=["id"])
        silver_df = spark.createDataFrame(silver_pdf)
        silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{self.target_db}.silver_{self.name}")
        
        self.metadata["completed_at"] = datetime.now().isoformat()
        print(f"✅ Complete: {self.metadata['records_ingested']} records, {self.metadata['pages_fetched']} pages")
        return self.metadata

# Run the pipeline
pipeline = APIIngestionPipeline("jsonplaceholder_posts", "https://jsonplaceholder.typicode.com/posts")
result = pipeline.ingest(max_pages=5, page_size=20)
print(f"\nMetadata: {json_lib.dumps(result, indent=2)}")

---
# 🏆 Section 11: Interview Questions

## Q1: How do you handle API rate limits?

1. **Respect Retry-After header** — sleep for the specified duration
2. **Exponential backoff** — double wait time on each 429
3. **Rate limiter** — track calls/minute, pre-emptively throttle
4. **Batch requests** — reduce total call count
5. **Cache responses** — avoid re-fetching unchanged data

## Q2: Incremental API ingestion?

Use a **high-water mark** (last_modified timestamp or cursor):
1. Store last sync timestamp in a metadata table
2. On next run: `GET /data?modified_since={last_sync}`
3. Only fetch new/changed records
4. Update watermark after successful ingestion

## Q3: How to make API ingestion idempotent?

1. **Idempotency keys** — deduplicate by event_id/record_id
2. **Upsert (MERGE)** — insert new, update existing
3. **Watermark tracking** — skip already-fetched pages
4. **Checkpointing** — resume from last successful page on failure

## Q4: Retry strategy for flaky APIs?

1. **Exponential backoff:** 1s → 2s → 4s → 8s (with jitter)
2. **Only retry retryable errors:** 429, 500, 502, 503, 504
3. **Don't retry client errors:** 400, 401, 403, 404
4. **Max retries:** 3-5 attempts, then fail and alert
5. **Circuit breaker:** After N consecutive failures, stop trying for a cooldown period

## Q5: OAuth2 token expiration in long pipelines?

1. Store token + expiry timestamp
2. Before each API call, check if token expires within 60s
3. If expiring: refresh token (POST to /oauth/token)
4. Use a wrapper class that auto-refreshes (see Section 4)
5. Handle refresh failures gracefully (retry refresh, then fail pipeline)

## Q6: Cursor vs offset pagination?

- **Offset:** Simple (`page=3&per_page=100`). Problem: if data changes between pages, you miss/duplicate records.
- **Cursor:** Stable (`cursor=abc123`). Server tracks position. No missed records even if data changes.
- **Prefer cursor** for: large datasets, frequently changing data, production pipelines.
- **Use offset** for: small static datasets, simple APIs, prototyping.

## Q7: Monitoring API ingestion?

Track per run: records fetched, errors, duration, pages, HTTP status distribution.
Alert on: error rate > threshold, zero records fetched, duration > 2x normal, 401/403 (auth broken).

## Q8: 20 different APIs with different auth?

1. **Config-driven:** Define each API in a config (url, auth_type, credentials_key, pagination_type)
2. **Factory pattern:** `APIClient.from_config(config)` creates the right client
3. **Shared retry/logging:** Common middleware for all clients
4. **Parallel execution:** Run independent APIs concurrently
5. **Centralized monitoring:** All pipelines report to same dashboard

---
# 📗 Section 5: Advanced Pagination Patterns

Pagination is the #1 challenge when ingesting from REST APIs. Different
APIs use different strategies — you need to handle all of them.

In [ ]:
import requests
import time
import json
from typing import Iterator, Optional, Dict, Any, List
from unittest.mock import patch, MagicMock

# ============================================================
# PATTERN 1: Offset/Limit Pagination
# ============================================================
def paginate_offset(base_url: str, params: dict = None,
                    page_size: int = 100, max_pages: int = 1000,
                    session: requests.Session = None) -> Iterator[dict]:
    """
    Handles offset/limit pagination.
    Yields individual records from all pages.
    
    Example API: GET /orders?limit=100&offset=0
    """
    sess = session or requests.Session()
    params = params or {}
    offset = 0
    pages_fetched = 0
    total_records = 0
    
    while pages_fetched < max_pages:
        page_params = {**params, "limit": page_size, "offset": offset}
        
        try:
            resp = sess.get(base_url, params=page_params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except requests.RequestException as e:
            print(f"  ⚠️  Page {pages_fetched + 1} failed: {e}")
            break
        
        # Handle different response shapes
        records = data if isinstance(data, list) else data.get("data", data.get("results", []))
        
        if not records:
            break  # No more data
        
        for record in records:
            yield record
        
        total_records += len(records)
        pages_fetched += 1
        offset += page_size
        
        print(f"  Page {pages_fetched}: {len(records)} records (total: {total_records})")
        
        # Check if we got a full page (if not, we're done)
        if len(records) < page_size:
            break
    
    print(f"  ✅ Pagination complete: {pages_fetched} pages, {total_records} records")

print("Offset pagination pattern defined.")
print("Usage: for record in paginate_offset(url, params={'status': 'active'}):")
print("           process(record)")

In [ ]:
# ============================================================
# PATTERN 2: Cursor-Based Pagination
# ============================================================
def paginate_cursor(base_url: str, params: dict = None,
                    cursor_field: str = "next_cursor",
                    data_field: str = "data",
                    session: requests.Session = None) -> Iterator[dict]:
    """
    Handles cursor-based pagination (used by Twitter, Stripe, etc.)
    
    Example response:
    {
        "data": [...],
        "next_cursor": "eyJpZCI6MTIzfQ==",
        "has_more": true
    }
    """
    sess = session or requests.Session()
    params = params or {}
    cursor = None
    page = 0
    total = 0
    
    while True:
        page_params = {**params}
        if cursor:
            page_params["cursor"] = cursor
        
        resp = sess.get(base_url, params=page_params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        
        records = data.get(data_field, [])
        if not records:
            break
        
        for record in records:
            yield record
        
        total += len(records)
        page += 1
        
        # Get next cursor
        cursor = data.get(cursor_field)
        has_more = data.get("has_more", bool(cursor))
        
        print(f"  Page {page}: {len(records)} records, cursor={cursor}")
        
        if not has_more or not cursor:
            break
    
    print(f"  ✅ Cursor pagination done: {page} pages, {total} records")

# ============================================================
# PATTERN 3: Link Header Pagination (GitHub-style)
# ============================================================
def paginate_link_header(base_url: str, headers: dict = None,
                         session: requests.Session = None) -> Iterator[dict]:
    """
    Follows Link headers for pagination (GitHub API, etc.)
    
    Response header: Link: <https://api.github.com/repos?page=2>; rel="next"
    """
    sess = session or requests.Session()
    url = base_url
    page = 0
    total = 0
    
    while url:
        resp = sess.get(url, headers=headers, timeout=30)
        resp.raise_for_status()
        
        records = resp.json()
        if not isinstance(records, list):
            records = records.get("items", records.get("data", []))
        
        for record in records:
            yield record
        
        total += len(records)
        page += 1
        
        # Parse Link header
        link_header = resp.headers.get("Link", "")
        next_url = None
        for part in link_header.split(","):
            if 'rel="next"' in part:
                next_url = part.split(";")[0].strip().strip("<>")
                break
        
        url = next_url
        print(f"  Page {page}: {len(records)} records, next={bool(next_url)}")
    
    print(f"  ✅ Link pagination done: {page} pages, {total} records")

print("All pagination patterns defined.")

## 5.2 Authentication Patterns

DE pipelines need to handle multiple auth schemes. Here are the
production patterns for each.

In [ ]:
# ============================================================
# AUTH PATTERN 1: API Key (simplest)
# ============================================================
class APIKeyAuth(requests.auth.AuthBase):
    """Custom auth class for API key authentication."""
    
    def __init__(self, api_key: str, header_name: str = "X-API-Key"):
        self.api_key = api_key
        self.header_name = header_name
    
    def __call__(self, r):
        r.headers[self.header_name] = self.api_key
        return r

# ============================================================
# AUTH PATTERN 2: OAuth2 Client Credentials (machine-to-machine)
# ============================================================
class OAuth2ClientCredentials:
    """
    OAuth2 client credentials flow for service-to-service auth.
    Automatically refreshes tokens before expiry.
    """
    
    def __init__(self, token_url: str, client_id: str, client_secret: str,
                 scope: str = None):
        self.token_url = token_url
        self.client_id = client_id
        self.client_secret = client_secret
        self.scope = scope
        self._token = None
        self._token_expiry = 0
    
    def get_token(self) -> str:
        """Get a valid token, refreshing if expired."""
        if self._token and time.time() < self._token_expiry - 60:
            return self._token  # Still valid (with 60s buffer)
        
        payload = {
            "grant_type": "client_credentials",
            "client_id": self.client_id,
            "client_secret": self.client_secret,
        }
        if self.scope:
            payload["scope"] = self.scope
        
        resp = requests.post(self.token_url, data=payload, timeout=30)
        resp.raise_for_status()
        token_data = resp.json()
        
        self._token = token_data["access_token"]
        self._token_expiry = time.time() + token_data.get("expires_in", 3600)
        print(f"  🔑 Token refreshed, expires in {token_data.get('expires_in', 3600)}s")
        return self._token
    
    def get_headers(self) -> dict:
        return {"Authorization": f"Bearer {self.get_token()}"}

# ============================================================
# AUTH PATTERN 3: HMAC Signature (webhook verification)
# ============================================================
import hmac
import hashlib

def verify_webhook_signature(payload: bytes, signature: str,
                              secret: str, algorithm: str = "sha256") -> bool:
    """Verify webhook payload signature (GitHub, Stripe pattern)."""
    expected = hmac.new(
        secret.encode(), payload, getattr(hashlib, algorithm)
    ).hexdigest()
    # Use constant-time comparison to prevent timing attacks
    return hmac.compare_digest(f"{algorithm}={expected}", signature)

# Test
secret = "my_webhook_secret"
payload = b'{"event": "order.created", "order_id": 123}'
sig = "sha256=" + hmac.new(secret.encode(), payload, hashlib.sha256).hexdigest()
print(f"Signature valid: {verify_webhook_signature(payload, sig, secret)}")
print(f"Tampered sig:    {verify_webhook_signature(payload, sig + 'x', secret)}")

## 5.3 Retry Logic & Circuit Breaker

Production API clients need resilience. Here's the full retry + circuit
breaker pattern used in enterprise DE pipelines.

In [ ]:
import random
from enum import Enum

class CircuitState(Enum):
    CLOSED = "closed"      # Normal operation
    OPEN = "open"          # Failing, reject requests
    HALF_OPEN = "half_open"  # Testing recovery

class ResilientAPIClient:
    """
    Production-grade API client with:
    - Exponential backoff with jitter
    - Circuit breaker pattern
    - Request/response logging
    - Timeout handling
    """
    
    def __init__(self, base_url: str, api_key: str = None,
                 max_retries: int = 3, base_delay: float = 1.0,
                 circuit_failure_threshold: int = 5,
                 circuit_recovery_timeout: float = 60.0):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        if api_key:
            self.session.headers["X-API-Key"] = api_key
        self.session.headers["Content-Type"] = "application/json"
        
        # Retry config
        self.max_retries = max_retries
        self.base_delay = base_delay
        
        # Circuit breaker state
        self.circuit_state = CircuitState.CLOSED
        self.failure_count = 0
        self.failure_threshold = circuit_failure_threshold
        self.last_failure_time = 0
        self.recovery_timeout = circuit_recovery_timeout
    
    def _check_circuit(self):
        """Check if circuit breaker allows the request."""
        if self.circuit_state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.circuit_state = CircuitState.HALF_OPEN
                print("  🔄 Circuit HALF_OPEN — testing recovery")
            else:
                raise Exception(f"Circuit OPEN — API unavailable, retry after {self.recovery_timeout}s")
    
    def _record_success(self):
        self.failure_count = 0
        if self.circuit_state == CircuitState.HALF_OPEN:
            self.circuit_state = CircuitState.CLOSED
            print("  ✅ Circuit CLOSED — API recovered")
    
    def _record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.circuit_state = CircuitState.OPEN
            print(f"  🔴 Circuit OPEN after {self.failure_count} failures")
    
    def get(self, endpoint: str, params: dict = None, **kwargs) -> dict:
        """GET with retry + circuit breaker."""
        self._check_circuit()
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        
        for attempt in range(self.max_retries + 1):
            try:
                resp = self.session.get(url, params=params, timeout=30, **kwargs)
                
                if resp.status_code == 429:  # Rate limited
                    retry_after = int(resp.headers.get("Retry-After", 60))
                    print(f"  ⏳ Rate limited, waiting {retry_after}s")
                    time.sleep(retry_after)
                    continue
                
                resp.raise_for_status()
                self._record_success()
                return resp.json()
            
            except requests.RequestException as e:
                self._record_failure()
                if attempt < self.max_retries:
                    # Exponential backoff with jitter
                    delay = self.base_delay * (2 ** attempt) + random.uniform(0, 1)
                    print(f"  ⚠️  Attempt {attempt+1} failed: {e}. Retrying in {delay:.1f}s")
                    time.sleep(delay)
                else:
                    raise

client = ResilientAPIClient("https://api.example.com", api_key="test-key")
print("ResilientAPIClient created")
print(f"Circuit state: {client.circuit_state.value}")
print(f"Max retries: {client.max_retries}")

---
# 📗 Section 6: Practice Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Build a Complete API Ingestion Pipeline
# ============================================================
# Simulate ingesting from a paginated API into a "Bronze" store

import json
from datetime import datetime, timezone

# Simulate API responses
def mock_api_server(page: int, page_size: int = 5):
    """Simulates a paginated API returning orders."""
    total_records = 23
    start = (page - 1) * page_size
    end = min(start + page_size, total_records)
    
    if start >= total_records:
        return {"data": [], "total": total_records, "has_more": False}
    
    records = [
        {
            "order_id": i + 1,
            "customer_id": (i % 5) + 1,
            "amount": round(50 + i * 17.3, 2),
            "status": ["pending", "completed", "shipped"][i % 3],
            "created_at": f"2024-01-{(i % 28) + 1:02d}T10:00:00Z",
        }
        for i in range(start, end)
    ]
    return {
        "data": records,
        "total": total_records,
        "page": page,
        "has_more": end < total_records,
    }

def ingest_orders_to_bronze(page_size: int = 5) -> list:
    """Ingest all orders from paginated API into Bronze layer."""
    bronze_records = []
    page = 1
    ingestion_time = datetime.now(timezone.utc).isoformat()
    
    while True:
        response = mock_api_server(page, page_size)
        records = response.get("data", [])
        
        if not records:
            break
        
        # Add Bronze metadata
        for record in records:
            bronze_record = {
                **record,
                "_ingested_at": ingestion_time,
                "_source": "orders_api",
                "_page": page,
                "_raw": json.dumps(record),
            }
            bronze_records.append(bronze_record)
        
        print(f"  Page {page}: {len(records)} records (total: {len(bronze_records)})")
        
        if not response.get("has_more"):
            break
        page += 1
    
    print(f"\n✅ Ingestion complete: {len(bronze_records)} records")
    return bronze_records

bronze = ingest_orders_to_bronze()
print(f"\nSample Bronze record:")
print(json.dumps(bronze[0], indent=2))

---
# 📗 Section 6: Production API Patterns

## Authentication Patterns

```python
# 1. API Key (most common)
headers = {"X-API-Key": "your-api-key-here"}
response = requests.get(url, headers=headers)

# 2. Bearer Token (OAuth2)
headers = {"Authorization": f"Bearer {access_token}"}

# 3. Basic Auth
from requests.auth import HTTPBasicAuth
response = requests.get(url, auth=HTTPBasicAuth("user", "password"))

# 4. OAuth2 with token refresh
def get_access_token(client_id, client_secret, token_url):
    response = requests.post(token_url, data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    })
    return response.json()["access_token"]
```

## Pagination Patterns

```python
# Pattern 1: Offset-based pagination
def fetch_all_offset(base_url, headers, page_size=100):
    all_records = []
    offset = 0
    while True:
        response = requests.get(base_url, headers=headers,
                               params={"limit": page_size, "offset": offset})
        data = response.json()
        records = data.get("results", [])
        all_records.extend(records)
        if len(records) < page_size:
            break  # Last page
        offset += page_size
    return all_records

# Pattern 2: Cursor-based pagination
def fetch_all_cursor(base_url, headers):
    all_records = []
    cursor = None
    while True:
        params = {"cursor": cursor} if cursor else {}
        response = requests.get(base_url, headers=headers, params=params)
        data = response.json()
        all_records.extend(data["data"])
        cursor = data.get("next_cursor")
        if not cursor:
            break
    return all_records

# Pattern 3: Link header pagination (GitHub-style)
def fetch_all_link_header(url, headers):
    all_records = []
    while url:
        response = requests.get(url, headers=headers)
        all_records.extend(response.json())
        # Parse Link header for next page
        link_header = response.headers.get("Link", "")
        url = None
        for part in link_header.split(","):
            if 'rel="next"' in part:
                url = part.split(";")[0].strip().strip("<>")
    return all_records
```

In [ ]:
# API patterns for data engineering
import requests
import time
from datetime import datetime

class APIClient:
    """Production-ready API client with retry, rate limiting, and pagination."""
    
    def __init__(self, base_url, api_key, rate_limit_per_second=10):
        self.base_url = base_url
        self.headers = {"X-API-Key": api_key, "Content-Type": "application/json"}
        self.rate_limit = rate_limit_per_second
        self.last_request_time = 0
        self.request_count = 0
    
    def _rate_limit(self):
        """Enforce rate limiting."""
        elapsed = time.time() - self.last_request_time
        min_interval = 1.0 / self.rate_limit
        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)
        self.last_request_time = time.time()
    
    def get(self, endpoint, params=None, max_retries=3):
        """GET with retry and rate limiting."""
        url = f"{self.base_url}/{endpoint}"
        
        for attempt in range(max_retries):
            self._rate_limit()
            try:
                response = requests.get(url, headers=self.headers,
                                       params=params, timeout=30)
                response.raise_for_status()
                self.request_count += 1
                return response.json()
            except requests.exceptions.HTTPError as e:
                if response.status_code == 429:  # Rate limited
                    retry_after = int(response.headers.get("Retry-After", 60))
                    print(f"   Rate limited. Waiting {retry_after}s...")
                    time.sleep(retry_after)
                elif response.status_code >= 500:  # Server error -- retry
                    if attempt < max_retries - 1:
                        time.sleep(2 ** attempt)
                    else:
                        raise
                else:
                    raise  # Client error -- don't retry
            except requests.exceptions.ConnectionError:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    raise
    
    def paginate(self, endpoint, page_size=100):
        """Fetch all pages of a paginated endpoint."""
        all_records = []
        page = 1
        while True:
            data = self.get(endpoint, params={"page": page, "per_page": page_size})
            records = data.get("results", data if isinstance(data, list) else [])
            all_records.extend(records)
            if len(records) < page_size:
                break
            page += 1
        return all_records


# Demo (simulated -- no real API call)
print("API Client Patterns for Data Engineering")
print("=" * 60)

print("\nKey patterns:")
patterns = [
    "Authentication: API key, Bearer token, OAuth2",
    "Retry: Exponential backoff for transient errors",
    "Rate limiting: Respect API limits (429 handling)",
    "Pagination: Offset, cursor, or Link header",
    "Timeout: Always set timeout (default: 30s)",
    "Error handling: Distinguish 4xx (client) vs 5xx (server)",
]
for p in patterns:
    print(f"  * {p}")

print("\nCommon DE API integrations:")
integrations = [
    "Salesforce (CRM data)",
    "Stripe (payment data)",
    "Google Analytics (web traffic)",
    "Slack (notifications)",
    "PagerDuty (alerting)",
    "GitHub (code metrics)",
]
for i in integrations:
    print(f"  * {i}")


In [ ]:
# ============================================
# 🎯 YOUR TURN: Build a Paginated API Fetcher
# ============================================
# Build a function that:
# 1. Fetches data from a paginated API (simulate with local data)
# 2. Handles rate limiting (sleep between requests)
# 3. Retries on failure (max 3 attempts)
# 4. Returns all records as a list
#
# Simulate the API with a local function that returns pages of data.


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
import time

# Simulate a paginated API
def mock_api(page, per_page=10):
    """Simulates a paginated API endpoint."""
    total_records = 35
    start = (page - 1) * per_page
    end = min(start + per_page, total_records)
    if start >= total_records:
        return {"results": [], "total": total_records, "page": page}
    return {
        "results": [{"id": i, "value": i * 10} for i in range(start + 1, end + 1)],
        "total": total_records,
        "page": page,
        "has_more": end < total_records
    }

def fetch_all_pages(api_fn, per_page=10, delay_seconds=0.1):
    """Fetch all pages from a paginated API."""
    all_records = []
    page = 1
    
    while True:
        # Simulate rate limiting
        time.sleep(delay_seconds)
        
        # Fetch page with retry
        for attempt in range(3):
            try:
                data = api_fn(page, per_page)
                break
            except Exception as e:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
        
        records = data.get("results", [])
        all_records.extend(records)
        
        print(f"   Page {page}: {len(records)} records (total so far: {len(all_records)})")
        
        if not data.get("has_more", False):
            break
        page += 1
    
    return all_records

print("Paginated API Fetcher")
print("=" * 50)
all_data = fetch_all_pages(mock_api, per_page=10)
print(f"\nTotal records fetched: {len(all_data)}")
print(f"First 3: {all_data[:3]}")
print(f"Last 3: {all_data[-3:]}")


---
# ✅ Notebook Complete!

**What was covered:**
- HTTP fundamentals: methods, status codes, headers
- requests library: GET, POST, params, headers, timeout
- Authentication: API keys, OAuth2 client credentials, token refresh
- Sessions: connection reuse, retry adapters
- Pagination: offset-based, cursor-based
- Error handling: exponential backoff, retryable vs non-retryable
- Batch ingestion: API → Bronze (raw JSON) → Silver (parsed, deduped)
- Webhooks: idempotency keys, batch processing
- Mini project: complete pipeline with metadata tracking
- 8 interview questions

**Next:** Notebook 13 — Logging for Data Engineering

---